<a href="https://colab.research.google.com/github/TomographicImaging/gVXR-Tutorials/blob/main/notebooks/magnification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
#
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:    Franck Vidal (UKRI-STFC)

![gVXR](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/Logo-transparent-small.png?raw=1)

# Magnification, SDD, SOD, ODD and inverse square law.

In this notebook, we will exploit the simulation data generated using [`magnification-simulation.ipynb`]
(./magnification-simulation.ipynb). We will
1. reconstruct the CT slices using the [core imaging library (CIL)](https://www.ccpi.ac.uk/cil/).
2. use the zero-mean, unit-variance normalisation to rescale pixel values so that all the images can be compared with
 each other regardless of their brightness and contrast.
3. identify imaging artefacts by comparing pairs of images:
  - smaller vs larger SDD for ice;
  - smaller vs larger SDD for aluminium; and
  - ice vs aluminium.
4. extract intensity profiles to assess noise and cupping artefacts.

![CT reconstructions for ice and aluminium at two different SDDs but with the same magnification](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/magnification4.png?raw=1)


<div class="alert alert-block alert-warning">
    <b>Note:</b> Make sure the Python packages are already installed. See <a href="../README.md">README.md</a> in the root directory of the repository. If you are running this notebook from Google Colab, please run the cell below to install the required packages.
</div>

Install condacolab if needed

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

Check if Conda is working well on Google Colab if needed

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import condacolab
    condacolab.check()

Install CIL using Conda on Google Colab if needed

In [ ]:
if IN_COLAB:
    !conda install -y -c conda-forge -c https://software.repos.intel.com/python/conda -c ccpi cil=24.3.0 ipp=2021.12 tigre

Install other system packages and gVXR


In [ ]:
if IN_COLAB:
    !apt-get install libnvidia-gl-580
    !apt-get install \
        libxcb-res0 \
        libxcb-ewmh2 \
        libxcb-composite0 \
        libxcb-cursor0 \
        libxcb-xinerama0 \
        libxcb-keysyms1 \
        libxcb-icccm4 \
        libxcb-xkb1

    !pip install gvxr

## Aims of this session

1. Know how to reconstruct data simulated in another notebook using the `JSON2gVXRDataReader` data reader for CIL.
2. Be able to choose a robust method to normalise images and allow a fair visual comparison.
3. Interpret images and intensity profiles to identify imaging artefacts.

![Intensity profiles marked in red above](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/magnification5.png?raw=1)

## Import packages

- `os` to create the output directory if needed
- `matplotlib` to show 2D images
- `SimpleITK` to load the DICOM file
- `gvxr` to simulate X-ray images
- `gvxr2json` to store details of a simulation in a JSON file
- Various `cil` packages for CT reconstruction

In [ ]:
# import numpy as np # Who does not use Numpy?

import matplotlib.pyplot as plt # Plotting
plt.style.use('tableau-colorblind10')

#  CT simulation
from gvxrPython3.JSON2gVXRDataReader import *

# CT reconstruction
from cil.plugins.astra import FBP
from cil.io import TIFFWriter

from cil.processors import TransmissionAbsorptionConverter
from cil.utilities.display import show_geometry

## Getting the data ready

Where to save the data.

In [ ]:
output_path = "./output_data/magnification"

# 1. Perform the CT reconstructions

In [ ]:
simulation_string_ID_set = [
    "material_H2O-smaller_SDD",
    "material_H2O-larger_SDD",
    "material_Al-smaller_SDD",
    "material_Al-larger_SDD"
]

reconstruction_set = []

for simulation_string_ID in simulation_string_ID_set:

    # Read the simulated data with CIL
    JSON_fname = os.path.join(output_path, simulation_string_ID + ".json")
    reader = JSON2gVXRDataReader(JSON_fname)
    data = reader.read()

    fig = show_geometry(data.geometry)
    fig.save(os.path.join(output_path, simulation_string_ID + "-geometry.png"))

    # Apply the minus log transformation (use use white_level=1.0 as the flat-field correction is already applied)
    data_corr = TransmissionAbsorptionConverter(white_level=1.0)(data)
    data_corr.reorder(order='astra')

    # We only want to reconstruct the slice in the middle of the volume (for the sake of speed)
    image_geometry = data_corr.geometry.get_ImageGeometry()

    # Comment if you want to reconstruct all the slices
    image_geometry.voxel_num_z = 1
    print("Image geometry", image_geometry)

    # Perform the reconstruction with CIL
    fbp = FBP(image_geometry, data_corr.geometry)
    fbp.set_input(data_corr)
    reconstruction = fbp.get_output()

    # Save the reconstructed CT images
    writer = TIFFWriter(data=reconstruction,
                        file_name=os.path.join(output_path, simulation_string_ID),
                        compression="uint16")
    writer.write()

    # Store the image
    reconstruction_set.append(reconstruction.array)

## 2. Normalise the images

We will use the zero-mean, unit-variance normalisation (also called standardisation or z-score normalisation in
machine learning) to allow a fair comparison. It is a common technique used in computer vision.

For an image $I$, each pixel value is transformed as:
$$I_{norm}(x,y) = \frac{I(x,y) - \mu(I)}{\sigma(I)}$$

Where:

- $I_{norm}(x,y)$ = original value at the location $(x,y)$;
- $\mu(I)$ = mean of the pixel values of Image $I$;
- $\sigma(I)$ = standard deviation of the pixel values of Image $I$.

After normalistion:

- $\mu(I_{norm})$ = 0
- $\sigma(I_{norm})$ = 1

This way, images can be visually compared with each other using the same visualisation window, i.e. to ensure they
all have the same brightness and contrast.
It avoids interpretation bias and ensures fairness.

Conversion to Numpy array

In [ ]:
reconstruction_set = np.array(reconstruction_set, dtype=np.single)

Perform the actual normalisation

In [ ]:
for image in reconstruction_set:
    image -= np.average(image)
    image /= np.std(image)

Display the images

In [ ]:
fig, axs = plt.subplots(2, len(simulation_string_ID_set) // 2, figsize=(10,10))

vmin = -2
vmax = 6.5
for i, (image, simulation_string_ID) in enumerate(zip(reconstruction_set, simulation_string_ID_set)):

    axs[i // 2, i % 2].set_title(simulation_string_ID)
    axs[i // 2, i % 2].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)

    axs[i // 2, i % 2].plot((150, 341), (255, 255), color="red")
plt.show()

Show the profiles marked in red above

In [ ]:
fig = plt.figure()

for i, (image, simulation_string_ID) in enumerate(zip(reconstruction_set, simulation_string_ID_set)):
    profile = image[255, 150:341]
    plt.plot(profile, label=simulation_string_ID)

plt.legend()
plt.show()

---
## Task:
1. Noise level:
  - Compare the two profiles for ice.
  - Compare the two profiles for aluminium.
  - What are your conclusions w.r.t. to noise and SDD?
2. Cupping artefacts:
  - Compare the profiles for ice and aluminium together.
  - The profiles of which sample are the flattest?
  - What are your conclusions?
3. There are another two types of artefacts that are more pronounced with aluminium.
    - They are related to each other.
    - What are they?